In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
import time

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Baseline Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso

# Tree-Based Models
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, 
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    StackingRegressor
)

# Advanced Boosting
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.style.use('default')
sns.set_palette('husl')

PREPROCESSED_DIR = Path('preprocessed_data')
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

print('Libraries loaded')

Libraries loaded


## 1. Feature Engineering

In [2]:
def add_features(df):
    """Add engineered features"""
    df = df.copy()
    
    year_col = None
    area_col = None
    rooms_col = None
    bath_col = None
    
    for col in df.columns:
        if 'year' in col.lower() or 'construction' in col.lower():
            year_col = col
        elif 'area' in col.lower() and 'total' in col.lower():
            area_col = col
        elif 'room' in col.lower() and 'bath' not in col.lower():
            rooms_col = col
        elif 'bath' in col.lower():
            bath_col = col
    
    if year_col:
        df['building_age'] = 2024 - df[year_col]
        df['building_age'] = df['building_age'].clip(0, 150)
        df['is_new'] = (df['building_age'] <= 5).astype(int)
        df['is_old'] = (df['building_age'] >= 50).astype(int)
    
    if rooms_col and area_col:
        df['room_density'] = df[rooms_col] / (df[area_col] + 1)
        df['area_per_room'] = df[area_col] / (df[rooms_col] + 1)
    
    if bath_col and rooms_col:
        df['total_amenities'] = df[rooms_col] + df[bath_col]
        df['bath_ratio'] = df[bath_col] / (df[rooms_col] + 1)
    
    if area_col:
        df['log_area'] = np.log1p(df[area_col])
        df['area_squared'] = df[area_col] ** 2
    
    return df

print('Feature engineering defined')

Feature engineering defined


## 2. Preprocessing Pipeline

In [3]:
class LocationEncoder(BaseEstimator, TransformerMixin):
    """Hierarchical country-region encoding"""
    
    def __init__(self, min_samples=100):
        self.min_samples = min_samples
        self.region_map_ = {}
    
    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        if 'country' not in X.columns or 'location' not in X.columns:
            return self
        
        for country in X['country'].unique():
            country_data = X[X['country'] == country]
            region_counts = country_data['location'].value_counts()
            common_regions = set(region_counts[region_counts >= self.min_samples].index)
            self.region_map_[country] = common_regions
        
        return self
    
    def transform(self, X):
        X = pd.DataFrame(X).copy()
        if 'country' not in X.columns or 'location' not in X.columns:
            return X
        
        def encode_row(row):
            country = row['country']
            location = row['location']
            
            if country not in self.region_map_:
                return f'rare_{country}'
            
            if location in self.region_map_[country]:
                return f'{country}_{location}'
            else:
                return f'rare_{country}'
        
        X['location_encoded'] = X.apply(encode_row, axis=1)
        X = X.drop(columns=['location'])
        return X


class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    """Group rare categories"""
    
    def __init__(self, min_freq=100):
        self.min_freq = min_freq
        self.keep_categories_ = {}
    
    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.keep_categories_ = {}
        
        for col in X.select_dtypes(include='object').columns:
            counts = X[col].value_counts()
            self.keep_categories_[col] = set(counts[counts >= self.min_freq].index)
        
        return self
    
    def transform(self, X):
        X = pd.DataFrame(X).copy()
        
        for col in X.select_dtypes(include='object').columns:
            if col in self.keep_categories_:
                keep = self.keep_categories_[col]
                X[col] = X[col].where(X[col].isin(keep), 'RARE')
        
        return X


def build_preprocessor(X, for_linear=False):
    """Build preprocessing pipeline
    
    for_linear: if True, adds StandardScaler for linear models
    """
    cat_cols = X.select_dtypes(include='object').columns.tolist()
    num_cols = X.select_dtypes(include=np.number).columns.tolist()
    
    if for_linear:
        num_pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ])
    else:
        num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median'))])
    
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('location', LocationEncoder(min_samples=100)),
        ('rare', RareCategoryGrouper(min_freq=100)),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ])
    
    transformers = []
    if num_cols:
        transformers.append(('num', num_pipe, num_cols))
    if cat_cols:
        transformers.append(('cat', cat_pipe, cat_cols))
    
    return ColumnTransformer(transformers, sparse_threshold=0)

print('Preprocessing defined')

Preprocessing defined


## 3. Sample Weighting for RMSE Optimization

In [4]:
def calculate_sample_weights(y):
    """Higher weights for expensive properties to reduce RMSE"""
    weights = np.sqrt(y / y.mean())
    weights = weights / weights.mean()
    return weights


def evaluate(y_true, y_pred):
    """Calculate metrics"""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1))) * 100
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape}

print('Evaluation functions defined')

Evaluation functions defined


## 4. Load and Prepare Data

In [5]:
# Use best preprocessing approach (minimal treatment - approach 3)
df = pd.read_csv(PREPROCESSED_DIR / 'real_estate_approach3_minimal_raw_top8.csv')
print(f'Loaded: {len(df):,} records, {len(df.columns)} columns')

# Feature engineering
df_fe = add_features(df)
print(f'After feature engineering: {len(df_fe.columns)} columns')

TARGET = 'price_in_USD'
X = df_fe.drop(columns=[TARGET])
y = df_fe[TARGET]

# Split: 70% train, 15% val, 15% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.176, random_state=RANDOM_STATE)

print(f'\nSplit sizes:')
print(f'  Train: {len(X_train):,} ({len(X_train)/len(df)*100:.1f}%)')
print(f'  Val:   {len(X_val):,} ({len(X_val)/len(df)*100:.1f}%)')
print(f'  Test:  {len(X_test):,} ({len(X_test)/len(df)*100:.1f}%)')

# Calculate sample weights
train_weights = calculate_sample_weights(y_train)
print(f'\nSample weights: min={train_weights.min():.3f}, mean={train_weights.mean():.3f}, max={train_weights.max():.3f}')

Loaded: 112,851 records, 12 columns
After feature engineering: 17 columns

Split sizes:
  Train: 79,040 (70.0%)
  Val:   16,883 (15.0%)
  Test:  16,928 (15.0%)

Sample weights: min=0.407, mean=1.000, max=2.381


### Price Snapshot (pre-training)
Quick distribution check of the target to spot extremes and center.

In [6]:
# Quick check of target price distribution
price_stats = df_fe[TARGET].agg(['min', 'median', 'max', 'mean'])
quantiles = df_fe[TARGET].quantile([0.05, 0.25, 0.5, 0.75, 0.95])

print('Price range (USD):')
print(f"  Min:   ${price_stats['min']:,.0f}")
print(f"  Median:${price_stats['median']:,.0f}")
print(f"  Max:   ${price_stats['max']:,.0f}")
print(f"  Mean:  ${price_stats['mean']:,.0f}")

print('\nKey quantiles (USD):')
for q, v in quantiles.items():
    print(f"  {int(q*100):>2}th: ${v:,.0f}")

Price range (USD):
  Min:   $37,934
  Median:$185,301
  Max:   $1,298,879
  Mean:  $265,735

Key quantiles (USD):
   5th: $56,992
  25th: $114,514
  50th: $185,301
  75th: $342,382
  95th: $754,141


## 5. Preprocessing for Different Model Types

In [7]:
# Preprocessor for linear models (with scaling)
preprocessor_linear = build_preprocessor(X_train, for_linear=True)
X_train_linear = preprocessor_linear.fit_transform(X_train, y_train)
X_val_linear = preprocessor_linear.transform(X_val)
X_test_linear = preprocessor_linear.transform(X_test)

print(f'Linear preprocessed shape: {X_train_linear.shape}')

# Preprocessor for tree models (no scaling)
preprocessor_tree = build_preprocessor(X_train, for_linear=False)
X_train_tree = preprocessor_tree.fit_transform(X_train, y_train)
X_val_tree = preprocessor_tree.transform(X_val)
X_test_tree = preprocessor_tree.transform(X_test)

print(f'Tree preprocessed shape: {X_train_tree.shape}')

Linear preprocessed shape: (79040, 16)
Tree preprocessed shape: (79040, 16)


## 6. Baseline Models (Linear, Ridge, Lasso)

In [8]:
baseline_models = {
    'Linear Regression': LinearRegression(),
    'Ridge (L2)': Ridge(alpha=100, random_state=RANDOM_STATE),
    'Lasso (L1)': Lasso(alpha=50, random_state=RANDOM_STATE, max_iter=5000)
}

baseline_results = {}

print('='*90)
print('BASELINE MODELS')
print('='*90)

for name, model in baseline_models.items():
    print(f'\n{name}:')
    start = time.time()
    
    # Train with weights
    model.fit(X_train_linear, y_train, sample_weight=train_weights)
    
    # Predictions
    pred_train = model.predict(X_train_linear)
    pred_val = model.predict(X_val_linear)
    pred_test = model.predict(X_test_linear)
    
    # Metrics
    train_metrics = evaluate(y_train, pred_train)
    val_metrics = evaluate(y_val, pred_val)
    test_metrics = evaluate(y_test, pred_test)
    
    elapsed = time.time() - start
    
    print(f'  Time: {elapsed:.2f}s')
    print(f'  TRAIN: MAE=${train_metrics["MAE"]:,.0f} | RMSE=${train_metrics["RMSE"]:,.0f} | R²={train_metrics["R2"]:.4f}')
    print(f'  VAL:   MAE=${val_metrics["MAE"]:,.0f} | RMSE=${val_metrics["RMSE"]:,.0f} | R²={val_metrics["R2"]:.4f}')
    print(f'  TEST:  MAE=${test_metrics["MAE"]:,.0f} | RMSE=${test_metrics["RMSE"]:,.0f} | R²={test_metrics["R2"]:.4f}')
    
    baseline_results[name] = {
        'model': model,
        'train': train_metrics,
        'val': val_metrics,
        'test': test_metrics,
        'time': elapsed
    }

print('\nBaseline models trained')

BASELINE MODELS

Linear Regression:
  Time: 0.03s
  TRAIN: MAE=$149,145 | RMSE=$195,803 | R²=0.2313
  VAL:   MAE=$161,948 | RMSE=$953,564 | R²=-16.4120
  TEST:  MAE=$148,316 | RMSE=$195,669 | R²=0.2296

Ridge (L2):
  Time: 0.02s
  TRAIN: MAE=$149,236 | RMSE=$195,787 | R²=0.2314
  VAL:   MAE=$162,020 | RMSE=$952,858 | R²=-16.3862
  TEST:  MAE=$148,408 | RMSE=$195,644 | R²=0.2297

Lasso (L1):
  Time: 3.65s
  TRAIN: MAE=$149,213 | RMSE=$195,789 | R²=0.2314
  VAL:   MAE=$161,935 | RMSE=$947,339 | R²=-16.1853
  TEST:  MAE=$148,384 | RMSE=$195,649 | R²=0.2297

Baseline models trained


## 7. Tree-Based Models

In [ ]:
tree_models = {
    'Random Forest': RandomForestRegressor(
        n_estimators=300,
        max_depth=20,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features='sqrt',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    
    'Extra Trees': ExtraTreesRegressor(
        n_estimators=300,
        max_depth=20,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features='sqrt',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=500,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=RANDOM_STATE
    ),
    
    'Hist Gradient Boosting': HistGradientBoostingRegressor(
        max_iter=500,
        max_depth=10,
        learning_rate=0.05,
        l2_regularization=1.0,
        random_state=RANDOM_STATE
    )
}

tree_results = {}

print('='*90)
print('TREE-BASED MODELS')
print('='*90)

for name, model in tree_models.items():
    print(f'\n{name}:')
    start = time.time()
    
    # Train with weights
    model.fit(X_train_tree, y_train, sample_weight=train_weights)
    
    # Predictions
    pred_train = model.predict(X_train_tree)
    pred_val = model.predict(X_val_tree)
    pred_test = model.predict(X_test_tree)
    
    # Metrics
    train_metrics = evaluate(y_train, pred_train)
    val_metrics = evaluate(y_val, pred_val)
    test_metrics = evaluate(y_test, pred_test)
    
    elapsed = time.time() - start
    
    print(f'  Time: {elapsed:.2f}s')
    print(f'  TRAIN: MAE=${train_metrics["MAE"]:,.0f} | RMSE=${train_metrics["RMSE"]:,.0f} | R²={train_metrics["R2"]:.4f}')
    print(f'  VAL:   MAE=${val_metrics["MAE"]:,.0f} | RMSE=${val_metrics["RMSE"]:,.0f} | R²={val_metrics["R2"]:.4f}')
    print(f'  TEST:  MAE=${test_metrics["MAE"]:,.0f} | RMSE=${test_metrics["RMSE"]:,.0f} | R²={test_metrics["R2"]:.4f}')
    
    tree_results[name] = {
        'model': model,
        'train': train_metrics,
        'val': val_metrics,
        'test': test_metrics,
        'time': elapsed
    }

print('\nTree-based models trained')

TREE-BASED MODELS

Random Forest:
  Time: 1.35s
  TRAIN: MAE=$56,688 | RMSE=$93,172 | R²=0.8259
  VAL:   MAE=$63,673 | RMSE=$105,534 | R²=0.7867
  TEST:  MAE=$62,737 | RMSE=$104,588 | R²=0.7799

Extra Trees:
  Time: 0.98s
  TRAIN: MAE=$77,409 | RMSE=$113,637 | R²=0.7411
  VAL:   MAE=$81,013 | RMSE=$120,046 | R²=0.7240
  TEST:  MAE=$79,699 | RMSE=$118,220 | R²=0.7188

Gradient Boosting:


## 8. Advanced Boosting Models (XGBoost, LightGBM, CatBoost)

In [ ]:
boosting_models = {
    'XGBoost': XGBRegressor(
        n_estimators=1000,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1.0,
        reg_lambda=1.0,
        min_child_weight=3,
        objective='reg:squarederror',
        eval_metric='rmse',
        random_state=RANDOM_STATE,
        n_jobs=-1,
        early_stopping_rounds=50
    ),
    
    'LightGBM': LGBMRegressor(
        n_estimators=1000,
        max_depth=10,
        learning_rate=0.05,
        num_leaves=50,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1.0,
        reg_lambda=1.0,
        min_child_samples=20,
        objective='rmse',
        metric='rmse',
        random_state=RANDOM_STATE,
        n_jobs=-1,
        force_col_wise=True
    ),
    
    'CatBoost': CatBoostRegressor(
        iterations=1000,
        depth=8,
        learning_rate=0.05,
        l2_leaf_reg=3.0,
        subsample=0.8,
        loss_function='RMSE',
        eval_metric='RMSE',
        random_state=RANDOM_STATE,
        verbose=0,
        early_stopping_rounds=50
    )
}

boosting_results = {}

print('='*90)
print('ADVANCED BOOSTING MODELS')
print('='*90)

for name, model in boosting_models.items():
    print(f'\n{name}:')
    start = time.time()
    
    # Train with early stopping
    if isinstance(model, XGBRegressor):
        model.fit(
            X_train_tree, y_train,
            sample_weight=train_weights,
            eval_set=[(X_val_tree, y_val)],
            verbose=False
        )
    elif isinstance(model, LGBMRegressor):
        model.fit(
            X_train_tree, y_train,
            sample_weight=train_weights,
            eval_set=[(X_val_tree, y_val)],
            callbacks=[]
        )
    elif isinstance(model, CatBoostRegressor):
        model.fit(
            X_train_tree, y_train,
            sample_weight=train_weights,
            eval_set=(X_val_tree, y_val),
            verbose=False
        )
    
    # Predictions
    pred_train = model.predict(X_train_tree)
    pred_val = model.predict(X_val_tree)
    pred_test = model.predict(X_test_tree)
    
    # Metrics
    train_metrics = evaluate(y_train, pred_train)
    val_metrics = evaluate(y_val, pred_val)
    test_metrics = evaluate(y_test, pred_test)
    
    elapsed = time.time() - start
    
    print(f'  Time: {elapsed:.2f}s')
    print(f'  TRAIN: MAE=${train_metrics["MAE"]:,.0f} | RMSE=${train_metrics["RMSE"]:,.0f} | R²={train_metrics["R2"]:.4f}')
    print(f'  VAL:   MAE=${val_metrics["MAE"]:,.0f} | RMSE=${val_metrics["RMSE"]:,.0f} | R²={val_metrics["R2"]:.4f}')
    print(f'  TEST:  MAE=${test_metrics["MAE"]:,.0f} | RMSE=${test_metrics["RMSE"]:,.0f} | R²={test_metrics["R2"]:.4f}')
    
    boosting_results[name] = {
        'model': model,
        'train': train_metrics,
        'val': val_metrics,
        'test': test_metrics,
        'time': elapsed
    }

print('\nAdvanced boosting models trained')

## 9. Stacking Ensemble (Combining Best Models)

In [ ]:
print('='*90)
print('STACKING ENSEMBLE')
print('='*90)

# Select best performing models as base estimators
# Use diverse model types for better generalization
base_estimators = [
    ('xgb', XGBRegressor(
        n_estimators=500, max_depth=8, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=1.0, reg_lambda=1.0,
        objective='reg:squarederror',
        random_state=RANDOM_STATE, n_jobs=-1
    )),
    ('lgb', LGBMRegressor(
        n_estimators=500, max_depth=10, learning_rate=0.05,
        num_leaves=50, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=1.0, reg_lambda=1.0,
        objective='rmse',
        random_state=RANDOM_STATE, n_jobs=-1, force_col_wise=True, verbose=-1
    )),
    ('cat', CatBoostRegressor(
        iterations=500, depth=8, learning_rate=0.05,
        l2_leaf_reg=3.0, loss_function='RMSE',
        random_state=RANDOM_STATE, verbose=0
    )),
    ('rf', RandomForestRegressor(
        n_estimators=200, max_depth=20, min_samples_split=10,
        random_state=RANDOM_STATE, n_jobs=-1
    )),
    ('et', ExtraTreesRegressor(
        n_estimators=200, max_depth=20, min_samples_split=10,
        random_state=RANDOM_STATE, n_jobs=-1
    ))
]

# Meta-learner: Ridge regression to combine predictions
meta_learner = Ridge(alpha=1.0)

# Create stacking ensemble
stacking_model = StackingRegressor(
    estimators=base_estimators,
    final_estimator=meta_learner,
    cv=5,
    n_jobs=-1
)

print('\nStacking Ensemble (5 base models + Ridge meta-learner):')
print('  Base models: XGBoost, LightGBM, CatBoost, Random Forest, Extra Trees')
print('  Meta-learner: Ridge Regression')
print('  CV folds: 5')

start = time.time()
stacking_model.fit(X_train_tree, y_train, sample_weight=train_weights)
elapsed = time.time() - start

# Predictions
pred_train = stacking_model.predict(X_train_tree)
pred_val = stacking_model.predict(X_val_tree)
pred_test = stacking_model.predict(X_test_tree)

# Metrics
train_metrics = evaluate(y_train, pred_train)
val_metrics = evaluate(y_val, pred_val)
test_metrics = evaluate(y_test, pred_test)

print(f'\n  Time: {elapsed:.2f}s')
print(f'  TRAIN: MAE=${train_metrics["MAE"]:,.0f} | RMSE=${train_metrics["RMSE"]:,.0f} | R²={train_metrics["R2"]:.4f}')
print(f'  VAL:   MAE=${val_metrics["MAE"]:,.0f} | RMSE=${val_metrics["RMSE"]:,.0f} | R²={val_metrics["R2"]:.4f}')
print(f'  TEST:  MAE=${test_metrics["MAE"]:,.0f} | RMSE=${test_metrics["RMSE"]:,.0f} | R²={test_metrics["R2"]:.4f}')

stacking_results = {
    'Stacking Ensemble': {
        'model': stacking_model,
        'train': train_metrics,
        'val': val_metrics,
        'test': test_metrics,
        'time': elapsed
    }
}

print('\nStacking ensemble trained')

## 10. Comprehensive Comparison Table

In [ ]:
# Combine all results
all_results = {
    **baseline_results,
    **tree_results,
    **boosting_results,
    **stacking_results
}

# Build comparison dataframe
comparison_data = []
for name, result in all_results.items():
    # Determine category
    if name in baseline_results:
        category = '1. Baseline'
    elif name in tree_results:
        category = '2. Tree-Based'
    elif name in boosting_results:
        category = '3. Advanced Boosting'
    else:
        category = '4. Stacking'
    
    comparison_data.append({
        'Category': category,
        'Model': name,
        'Test_MAE': result['test']['MAE'],
        'Test_RMSE': result['test']['RMSE'],
        'Test_R2': result['test']['R2'],
        'Test_MAPE': result['test']['MAPE'],
        'Val_RMSE': result['val']['RMSE'],
        'Train_RMSE': result['train']['RMSE'],
        'Overfit': result['train']['RMSE'] - result['test']['RMSE'],
        'Time_s': result['time']
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('Test_RMSE')

print('\n' + '='*140)
print('COMPLETE MODEL COMPARISON (sorted by Test RMSE)')
print('='*140)
print(comparison_df.to_string(index=False))

# Best model
best_idx = comparison_df['Test_RMSE'].idxmin()
best = comparison_df.loc[best_idx]

print(f'\n{"="*90}')
print('🏆 BEST MODEL')
print('='*90)
print(f'Model:      {best["Model"]}')
print(f'Category:   {best["Category"]}')
print(f'Test RMSE:  ${best["Test_RMSE"]:,.0f}')
print(f'Test MAE:   ${best["Test_MAE"]:,.0f}')
print(f'Test R²:    {best["Test_R2"]:.4f}')
print(f'Test MAPE:  {best["Test_MAPE"]:.2f}%')
print(f'Time:       {best["Time_s"]:.1f}s')

# Save comparison
comparison_df.to_csv(MODEL_DIR / 'complete_model_comparison.csv', index=False)
print(f'\nSaved comparison to: {MODEL_DIR / "complete_model_comparison.csv"}')

# Save best model
best_model_name = best['Model']
best_model = all_results[best_model_name]['model']

if best_model_name in baseline_results:
    best_preprocessor = preprocessor_linear
else:
    best_preprocessor = preprocessor_tree

joblib.dump({
    'model': best_model,
    'preprocessor': best_preprocessor,
    'metrics': all_results[best_model_name],
    'model_name': best_model_name
}, MODEL_DIR / 'best_model_final.joblib')

print(f'Saved best model to: {MODEL_DIR / "best_model_final.joblib"}')

## 11. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 1. RMSE comparison by category
comparison_sorted = comparison_df.sort_values('Test_RMSE')
colors = {'1. Baseline': 'lightcoral', '2. Tree-Based': 'lightblue', 
          '3. Advanced Boosting': 'lightgreen', '4. Stacking': 'gold'}
bar_colors = [colors[cat] for cat in comparison_sorted['Category']]

axes[0, 0].barh(comparison_sorted['Model'], comparison_sorted['Test_RMSE'], 
                color=bar_colors, edgecolor='black')
axes[0, 0].set_xlabel('Test RMSE ($)', fontsize=12)
axes[0, 0].set_title('Test RMSE by Model (Lower is Better)', fontweight='bold', fontsize=13)
axes[0, 0].grid(alpha=0.3, axis='x')
axes[0, 0].invert_yaxis()

# 2. MAE vs RMSE
for category, color in colors.items():
    mask = comparison_df['Category'] == category
    axes[0, 1].scatter(comparison_df[mask]['Test_MAE'], 
                       comparison_df[mask]['Test_RMSE'],
                       s=150, alpha=0.7, color=color, edgecolor='black',
                       label=category.split('. ')[1])

axes[0, 1].scatter(best['Test_MAE'], best['Test_RMSE'], 
                   s=400, marker='*', color='red', edgecolor='black', 
                   linewidth=2, label='Best', zorder=10)
axes[0, 1].set_xlabel('Test MAE ($)', fontsize=12)
axes[0, 1].set_ylabel('Test RMSE ($)', fontsize=12)
axes[0, 1].set_title('MAE vs RMSE Trade-off', fontweight='bold', fontsize=13)
axes[0, 1].legend(loc='best')
axes[0, 1].grid(alpha=0.3)

# 3. R² comparison
axes[0, 2].barh(comparison_sorted['Model'], comparison_sorted['Test_R2'], 
                color=bar_colors, edgecolor='black')
axes[0, 2].set_xlabel('Test R² (Closer to 1 is Better)', fontsize=12)
axes[0, 2].set_title('Test R² by Model', fontweight='bold', fontsize=13)
axes[0, 2].grid(alpha=0.3, axis='x')
axes[0, 2].invert_yaxis()

# 4. Training time comparison
axes[1, 0].barh(comparison_sorted['Model'], comparison_sorted['Time_s'], 
                color=bar_colors, edgecolor='black')
axes[1, 0].set_xlabel('Training Time (seconds)', fontsize=12)
axes[1, 0].set_title('Training Time by Model', fontweight='bold', fontsize=13)
axes[1, 0].grid(alpha=0.3, axis='x')
axes[1, 0].invert_yaxis()

# 5. Overfitting analysis
axes[1, 1].barh(comparison_sorted['Model'], comparison_sorted['Overfit'], 
                color=bar_colors, edgecolor='black')
axes[1, 1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1, 1].set_xlabel('Train RMSE - Test RMSE ($)', fontsize=12)
axes[1, 1].set_title('Overfitting Analysis (Negative = Better Test)', fontweight='bold', fontsize=13)
axes[1, 1].grid(alpha=0.3, axis='x')
axes[1, 1].invert_yaxis()

# 6. Category summary
category_summary = comparison_df.groupby('Category').agg({
    'Test_RMSE': 'mean',
    'Test_MAE': 'mean',
    'Test_R2': 'mean'
}).reset_index()

x_pos = np.arange(len(category_summary))
width = 0.25

axes[1, 2].bar(x_pos - width, category_summary['Test_RMSE']/1000, width, 
               label='RMSE (K$)', color='coral', edgecolor='black')
axes[1, 2].bar(x_pos, category_summary['Test_MAE']/1000, width, 
               label='MAE (K$)', color='steelblue', edgecolor='black')
axes[1, 2].bar(x_pos + width, category_summary['Test_R2']*100, width, 
               label='R² (×100)', color='lightgreen', edgecolor='black')

axes[1, 2].set_xticks(x_pos)
axes[1, 2].set_xticklabels([cat.split('. ')[1] for cat in category_summary['Category']], 
                            rotation=45, ha='right')
axes[1, 2].set_ylabel('Value', fontsize=12)
axes[1, 2].set_title('Average Performance by Category', fontweight='bold', fontsize=13)
axes[1, 2].legend()
axes[1, 2].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(MODEL_DIR / 'complete_model_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

print(f'Saved visualization to: {MODEL_DIR / "complete_model_comparison.png"}')

## 12. Feature Importance (Best Model)

In [ ]:
# Get feature importance if available
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    
    # Get feature names
    if best_model_name in baseline_results:
        feature_names = [f'Feature_{i}' for i in range(len(importances))]
    else:
        feature_names = [f'Feature_{i}' for i in range(len(importances))]
    
    # Sort by importance
    indices = np.argsort(importances)[::-1][:20]  # Top 20
    
    plt.figure(figsize=(12, 8))
    plt.barh(range(len(indices)), importances[indices], color='steelblue', edgecolor='black')
    plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
    plt.xlabel('Importance', fontsize=12)
    plt.title(f'Top 20 Feature Importances - {best_model_name}', fontweight='bold', fontsize=14)
    plt.gca().invert_yaxis()
    plt.grid(alpha=0.3, axis='x')
    plt.tight_layout()
    plt.savefig(MODEL_DIR / 'feature_importance.png', dpi=200, bbox_inches='tight')
    plt.show()
    
    print(f'Saved feature importance to: {MODEL_DIR / "feature_importance.png"}')
else:
    print(f'{best_model_name} does not have feature_importances_ attribute')

## 13. Hybrid Model: Country-Specific + Global Fallback
Combine per-country specialists with the best global model, and compare against the current champion.

In [ ]:
print('='*100)
print('HYBRID MODEL (country specialists + global fallback)')
print('='*100)

# Base data
df_hybrid = df_fe.copy()
country_counts = df_hybrid['country'].value_counts()
min_samples = 500
major_countries = country_counts[country_counts >= min_samples].index.tolist()

print(f'Countries with ≥{min_samples} samples: {len(major_countries)}')
if len(major_countries) > 0:
    print(country_counts[country_counts >= min_samples].to_string())
else:
    print('No countries meet the minimum sample threshold; hybrid will fall back to the global model only.')

# Helper: choose a strong single-model type for country specialists
def create_country_model(global_best_name):
    if 'XGBoost' in global_best_name:
        return XGBRegressor(
            n_estimators=800, max_depth=8, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            reg_alpha=1.0, reg_lambda=1.0,
            objective='reg:squarederror',
            random_state=RANDOM_STATE, n_jobs=-1,
            early_stopping_rounds=50, eval_metric='rmse'
        )
    if 'LightGBM' in global_best_name:
        return LGBMRegressor(
            n_estimators=800, max_depth=10, learning_rate=0.05,
            num_leaves=50, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=1.0, reg_lambda=1.0,
            min_child_samples=20,
            objective='rmse', metric='rmse',
            random_state=RANDOM_STATE, n_jobs=-1, force_col_wise=True
        )
    if 'CatBoost' in global_best_name:
        return CatBoostRegressor(
            iterations=800, depth=8, learning_rate=0.05,
            l2_leaf_reg=3.0, loss_function='RMSE',
            eval_metric='RMSE',
            random_state=RANDOM_STATE, verbose=0,
            early_stopping_rounds=50
        )
    # Fallback
    return XGBRegressor(
        n_estimators=800, max_depth=8, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=1.0, reg_lambda=1.0,
        objective='reg:squarederror',
        random_state=RANDOM_STATE, n_jobs=-1,
        early_stopping_rounds=50, eval_metric='rmse'
    )

# Helper: train a country model with RMSE-focused weighting
def train_country_model(model, X_train, y_train, X_val, y_val, X_test, y_test, label):
    start = time.time()
    train_weights = calculate_sample_weights(y_train)

    if isinstance(model, XGBRegressor):
        model.fit(X_train, y_train, sample_weight=train_weights, eval_set=[(X_val, y_val)], verbose=False)
    elif isinstance(model, LGBMRegressor):
        model.fit(X_train, y_train, sample_weight=train_weights, eval_set=[(X_val, y_val)], callbacks=[])
    elif isinstance(model, CatBoostRegressor):
        model.fit(X_train, y_train, sample_weight=train_weights, eval_set=(X_val, y_val), verbose=False)
    else:
        model.fit(X_train, y_train, sample_weight=train_weights)

    preds_train = model.predict(X_train)
    preds_val = model.predict(X_val)
    preds_test = model.predict(X_test)

    metrics = {
        'train': evaluate(y_train, preds_train),
        'val': evaluate(y_val, preds_val),
        'test': evaluate(y_test, preds_test)
    }
    elapsed = time.time() - start
    print(f"{label}: RMSE test=${metrics['test']['RMSE']:,.0f} | time={elapsed:.1f}s")
    return model, metrics, elapsed

country_models = {}
country_results = []

for country in major_countries:
    df_country = df_hybrid[df_hybrid['country'] == country].copy()
    if len(df_country) < 300:
        print(f'Skipping {country}: only {len(df_country)} samples')
        continue

    X_c = df_country.drop(columns=[TARGET])
    y_c = df_country[TARGET]

    X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_c, y_c, test_size=0.2, random_state=RANDOM_STATE)
    X_train_c, X_val_c, y_train_c, y_val_c = train_test_split(X_train_c, y_train_c, test_size=0.2, random_state=RANDOM_STATE)

    preprocessor_c = build_preprocessor(X_train_c)
    X_train_c_prep = preprocessor_c.fit_transform(X_train_c, y_train_c)
    X_val_c_prep = preprocessor_c.transform(X_val_c)
    X_test_c_prep = preprocessor_c.transform(X_test_c)

    model = create_country_model(best_model_name)
    trained_model, metrics, elapsed = train_country_model(
        model, X_train_c_prep, y_train_c, X_val_c_prep, y_val_c, X_test_c_prep, y_test_c,
        f'{country} model'
    )

    country_models[country] = {
        'model': trained_model,
        'preprocessor': preprocessor_c,
        'metrics': metrics
    }
    country_results.append({
        'Country': country,
        'Samples': len(df_country),
        'Test_RMSE': metrics['test']['RMSE'],
        'Test_MAE': metrics['test']['MAE'],
        'Test_R2': metrics['test']['R2']
    })

if country_results:
    country_results_df = pd.DataFrame(country_results).sort_values('Test_RMSE')
    print('\nCountry specialist summary (sorted by RMSE):')
    print(country_results_df.to_string(index=False))
else:
    print('\nNo country-specific models were trained.')

# Hybrid predictor
class HybridPredictor:
    """Use country model when available, otherwise fall back to global model"""
    def __init__(self, country_models, global_model, global_preprocessor):
        self.country_models = country_models
        self.global_model = global_model
        self.global_preprocessor = global_preprocessor
    def predict(self, X):
        X = X.copy()
        preds = np.zeros(len(X))
        for country, model_data in self.country_models.items():
            mask = X['country'] == country
            if mask.any():
                X_country = X[mask]
                X_prep = model_data['preprocessor'].transform(X_country)
                preds[mask] = model_data['model'].predict(X_prep)
        other_mask = ~X['country'].isin(self.country_models.keys())
        if other_mask.any():
            X_other = X[other_mask]
            X_prep = self.global_preprocessor.transform(X_other)
            preds[other_mask] = self.global_model.predict(X_prep)
        return preds

# Use the already-trained global best model
global_model = best_model
global_preprocessor = best_preprocessor
global_rmse = best['Test_RMSE']

hybrid = HybridPredictor(country_models, global_model, global_preprocessor)

# Evaluate hybrid on a fresh split
X_full = df_hybrid.drop(columns=[TARGET])
y_full = df_hybrid[TARGET]

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_full, y_full, test_size=0.2, random_state=RANDOM_STATE)
y_pred_hybrid = hybrid.predict(X_test_h)
hybrid_metrics = evaluate(y_test_h, y_pred_hybrid)

print('\nHybrid vs Global:')
print(f"Global RMSE: ${global_rmse:,.0f}")
print(f"Hybrid RMSE: ${hybrid_metrics['RMSE']:,.0f}")
improvement = ((global_rmse - hybrid_metrics['RMSE']) / global_rmse) * 100
print(f"Improvement: {improvement:+.2f}%")
print(f"Test MAE:  ${hybrid_metrics['MAE']:,.0f}")
print(f"Test R²:   {hybrid_metrics['R2']:.4f}")
print(f"Test MAPE: {hybrid_metrics['MAPE']:.2f}%")

# Save hybrid
hybrid_payload = {
    'hybrid_predictor': hybrid,
    'country_models': country_models,
    'global_model': global_model,
    'global_preprocessor': global_preprocessor,
    'metrics': hybrid_metrics
}
joblib.dump(hybrid_payload, MODEL_DIR / 'hybrid_country_model.joblib')
print(f"Saved hybrid model to: {MODEL_DIR / 'hybrid_country_model.joblib'}")

## Summary

### Models Tested:
1. **Baseline Models**: Linear Regression, Ridge, Lasso
2. **Tree-Based**: Random Forest, Extra Trees, Gradient Boosting, Hist Gradient Boosting
3. **Advanced Boosting**: XGBoost, LightGBM, CatBoost (RMSE-optimized)
4. **Stacking Ensemble**: 5 base models + Ridge meta-learner

### Optimization Techniques:
- Sample weighting for expensive properties
- RMSE-specific objectives
- Strong regularization (L1, L2)
- Early stopping for boosting models
- Cross-validated stacking

### Results:
- See comparison table above
- Best model saved to `models/best_model_final.joblib`
- Full comparison in `models/complete_model_comparison.csv`